# Извлечение незарегистрированных спам-сообщений

Логика: из таблицы `collected_message` (все сообщения, которые видел бот) вычитаем:
1. Тексты из экспорта чата (сообщения, которые до сих пор в чате = безопасные)
2. Тексты из таблицы `spam_message` (уже пойманный спам)

Остаток = сообщения, которые были удалены из чата, но не были пойманы ботом.  
С высокой вероятностью это спам, удалённый вручную модераторами.

## Импорт необходимых библиотек

In [1]:
import json
import os
import sqlite3
import sys

import pandas as pd

## Конфигурация путей

In [3]:
BASE_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else os.path.abspath('..')

In [4]:
DB_PATH = os.path.join(BASE_DIR, 'instance', 'db.sqlite')

In [5]:
CHAT_EXPORT_PATH = os.path.join(BASE_DIR, 'chat_exports', 'ChatExport_STANKIN_Abiturient_2026-03-25_texts.json')

In [6]:
OUTPUT_PATH = os.path.join(BASE_DIR, 'data', 'unregistered_spam.json')

## Загрузка данных

### Collected messages из БД

In [7]:
conn = sqlite3.connect(DB_PATH)

In [8]:
df_collected = pd.read_sql_query("SELECT DISTINCT message_text FROM collected_message", conn)

### Spam messages из БД

In [9]:
df_spam = pd.read_sql_query("SELECT DISTINCT message_text FROM spam_message", conn)

In [10]:
conn.close()

### Chat export (безопасные сообщения из чата)

In [11]:
if os.path.exists(CHAT_EXPORT_PATH):
    with open(CHAT_EXPORT_PATH, 'r', encoding='utf-8') as f:
        chat_data = json.load(f)
    chat_texts = set(d['text'] for d in chat_data if d.get('text') and d['text'].strip())
    print(f"Chat export texts (non-empty): {len(chat_texts)}")
else:
    chat_texts = set()
    print("Chat export не найден – пропускаем")

Chat export texts (non-empty): 746


## Вычисление: collected - chat_export - spam

In [12]:
collected_texts = set(df_collected['message_text'].tolist())

In [13]:
spam_texts = set(df_spam['message_text'].tolist())

In [15]:
unregistered = collected_texts - chat_texts - spam_texts

In [19]:
print(f"\nСобрано:            {len(collected_texts)}")
print(f"Экспорт чатов:      {len(chat_texts)}")
print(f"Спам (поймано):     {len(spam_texts)}")
print(f"Незарегистрировано: {len(unregistered)}")


Собрано:            285
Экспорт чатов:      746
Спам (поймано):     668
Незарегистрировано: 33


## Просмотр результатов

In [20]:
if unregistered:
    print(f"\nНезарегистрированные спам-сообщения ({len(unregistered)}):\n")
    for i, text in enumerate(sorted(unregistered, key=len, reverse=True), 1):
        preview = text[:120].replace('\n', ' ')
        print(f"{i:3}. [{len(text):>4} символов] {preview}")
else:
    print("Незарегистрированных спам-сообщений не найдено")


Незарегистрированные спам-сообщения (33):

  1. [ 599 символов] ⚡️⚡️⚡️  Дорогие родители, напоминаем вам, что ровно через час в 19:00 у нас состоится Родительское собрание с Ответствен
  2. [ 502 символов] 🥰Помимо участия в Дне открытых дверей, наша команда в течение прошлых выходных также принимала участие в выставке Навига
  3. [ 480 символов] Ой апять какая то арава заумных слов 🙄 эх вы ученые маченые капчёные все думаете ищете чего то а как пасматреть на эти в
  4. [ 435 символов] Прикольно, что даже такие важные моменты как прием в учебные заведения можно связывать со знаками зодиака. Это напоминае
  5. [ 399 символов] Интересный подход к учебным правилам с точки зрения астрологии. Почему бы не посмотреть на свою натальную карту перед по
  6. [ 378 символов] 🚀 7 февраля МГТУ «СТАНКИН» стал ключевой площадкой городского «Посвящения в будущие профессионалы».   Школьники инженерн
  7. [ 375 символов] Прикольно, как астрология проникает в повседневную жизнь, даже в подготовку к ЕГЭ! 

## Сохранение в формате датасета

In [21]:
if unregistered:
    unregistered_data = [{"text": text, "label": 1} for text in unregistered if text.strip()]
    
    with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
        json.dump(unregistered_data, f, ensure_ascii=False, indent=4)
    
    print(f"\nСохранено {len(unregistered_data)} записей в {OUTPUT_PATH}")
    print("Этот файл можно подключить в 1. Подготовка данных.ipynb как дополнительный источник спама")
else:
    print("Нечего сохранять")


Сохранено 33 записей в C:\Users\overklassniy\PycharmProjects\STANKIN_AntiSpam_Bot\data\unregistered_spam.json
Этот файл можно подключить в 1. Подготовка данных.ipynb как дополнительный источник спама
